# 06 — Middleware: Intercepting Agent Behaviour

Middleware lets you plug behaviour into the agent loop **without** rewriting the agent. Two of the most useful built-ins:

1. **`SummarizationMiddleware`** — auto-compresses old conversation history when it gets long.
2. **`HumanInTheLoopMiddleware`** — pauses the agent before a tool fires so a human can approve / edit / reject the call.

Other things middleware is good for: logging & analytics, prompt rewriting, retry/fallback logic, rate limiting, PII redaction, guardrails.

**Provider:** `groq:qwen/qwen3-32b` for everything, including the summariser's own model.

> Read [`06_middleware.md`](./06_middleware.md) alongside.


## Setup

Both middleware types need a **checkpointer** so state can persist across `agent.invoke(...)` calls. We use `InMemorySaver` here — fine for notebooks. In production you'd swap for `SqliteSaver`, `PostgresSaver`, etc.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY missing from .env"


---

## Part 1 — `SummarizationMiddleware`

Without summarisation, every turn of an agent's history is sent back to the model. Long-running conversations blow through the context window and cost more per call.

`SummarizationMiddleware` watches the message history and, when a configurable **trigger** fires, runs the summarisation model on the oldest messages and replaces them with a single summary message. Recent messages stay verbatim so the agent doesn't lose immediate context.

Three trigger styles:

| Trigger | Example | Fires when |
|---|---|---|
| `("messages", N)` | `("messages", 10)` | message count ≥ N |
| `("tokens", N)` | `("tokens", 550)` | token count ≥ N |
| `("fraction", f)` | `("fraction", 0.02)` | token count ≥ f × context window |

`keep=` uses the same shape and controls how much **recent** history to preserve verbatim after summarisation.


### 1.1 — Message-based trigger

Easiest to understand: count messages, trigger when there are too many. We trigger at 10 messages and keep the last 4.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)

# Same thread_id across calls means messages accumulate in one conversation.
config = {"configurable": {"thread_id": "summarise-msgs"}}


Watch the message count climb past the trigger, then drop after summarisation fires:


In [ ]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"After {q!r:18}  →  {len(response['messages'])} messages total")


### 1.2 — Token-based trigger

More precise than message count for varying message sizes. Trigger at 550 tokens, keep 200 tokens of recent history.

We add a `search_hotels` tool that returns chunky responses so the token count grows quickly.


In [ ]:
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns a long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ],
)

config = {"configurable": {"thread_id": "summarise-tokens"}}


def count_tokens(messages):
    """Rough estimate: 4 chars ≈ 1 token."""
    return sum(len(str(m.content)) for m in messages) // 4


In [ ]:
cities = ["Paris", "London"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config,
    )
    print(f"{city:8}  →  ~{count_tokens(response['messages']):4} tokens, {len(response['messages'])} messages")


### 1.3 — Fraction-based trigger

Define the trigger as a fraction of the model's context window. Useful when you want the same config to scale across models with different context sizes.

`qwen/qwen3-32b` has a 32,768-token context window. `0.02` ≈ 655 tokens.


In [ ]:
QWEN_CONTEXT = 32_768  # qwen/qwen3-32b context window

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.02),
            keep=("fraction", 0.008),
        ),
    ],
)

config = {"configurable": {"thread_id": "summarise-fraction"}}

cities = ["Paris", "London", "Tokyo", "New York"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config,
    )
    tokens = count_tokens(response['messages'])
    fraction = tokens / QWEN_CONTEXT
    print(f"{city:10}  →  ~{tokens:4} tokens ({fraction:.4%}), {len(response['messages'])} msgs")


---

## Part 2 — `HumanInTheLoopMiddleware`

Some tool calls should not run silently. Database writes, financial transactions, outbound emails — anything irreversible or expensive needs human eyes first.

`HumanInTheLoopMiddleware` pauses the agent **right before** a configured tool fires, returns an `__interrupt__` value, and waits. Your code reviews the proposed call, then resumes the agent with one of three decisions:

| Decision | What happens |
|---|---|
| `approve` | Tool runs with the model's original args. |
| `edit` | Tool runs with the args **you** specify in `edited_action`. |
| `reject` | Tool does not run. The agent sees a `ToolMessage` with `status='error'` and continues. |

`interrupt_on={tool_name: ...}` configures which tools require approval. Set a tool to `False` to skip the interrupt and let it run freely.


### 2.1 — Approve flow

We register two mock tools — `read_email_tool` (safe, no interrupt) and `send_email_tool` (interrupts before running).


In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command


def read_email_tool(email_id: str) -> str:
    """Mock: read an email by its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock: send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,  # no interrupt — runs freely
            }
        )
    ],
)


**Step 1 — invoke and observe the interrupt.** The model decides to call `send_email_tool`, but the middleware pauses the graph and returns `__interrupt__` in the response.


In [ ]:
config = {"configurable": {"thread_id": "test-approve"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config,
)

print("Interrupted?:", "__interrupt__" in result)
print("Pending tool call:", result["messages"][-1].tool_calls)


**Step 2 — resume with `approve`.** The pending tool call executes with its original arguments; the agent finishes normally.


In [ ]:
if "__interrupt__" in result:
    print("⏸️  Paused. Approving the call...")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,
    )
    print(f"✅ Final answer: {result['messages'][-1].content}")


### 2.2 — Reject flow

Same setup; this time we deny the call. The agent sees an error `ToolMessage` and reasons over it (typically asking the user what to do next).


In [ ]:
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
                "read_email_tool": False,
            }
        ),
    ],
)

config = {"configurable": {"thread_id": "test-reject"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config,
)


In [ ]:
if "__interrupt__" in result:
    print("⏸️  Paused. Rejecting the call...")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject"}]}),
        config=config,
    )
    print(f"❌ Final answer: {result['messages'][-1].content}")


### 2.3 — Edit flow

The most powerful path: the model proposes a tool call, you **rewrite the arguments**, then execute. Useful when the model gets the recipient / amount / id slightly wrong and you want to fix it without restarting the conversation.


In [ ]:
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
                "read_email_tool": False,
            }
        ),
    ],
)

config = {"configurable": {"thread_id": "test-edit"}}

# Note the deliberately wrong recipient — we'll fix it on resume.
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config,
)

print("Model proposed args:", result["messages"][-1].tool_calls[0]["args"])


In [ ]:
if "__interrupt__" in result:
    print("⏸️  Paused. Editing the call args before approving...")
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by a human before sending",
                            },
                        },
                    }
                ]
            }
        ),
        config=config,
    )
    print(f"✏️  Final answer: {result['messages'][-1].content}")


---

## Recap

- **Middleware = behaviour plugins for the agent loop.** No rewriting the agent.
- **`SummarizationMiddleware`** keeps long conversations within the context window. Three triggers — `messages`, `tokens`, `fraction` — pick the one that matches how you want to budget context.
- **`HumanInTheLoopMiddleware`** pauses the agent before risky tool calls and waits for your `approve` / `edit` / `reject` decision. Resume via `Command(resume={...})`.
- Both middleware require a **checkpointer** (`InMemorySaver` for notebooks; production-grade savers for real apps).
- A **`thread_id` in config** identifies a conversation thread — same id across invokes = same conversation, different ids = independent threads.

**Course complete.** Where to go from here is listed at the bottom of [`README.md`](./README.md): structured output, RAG, LangGraph state machines, LangSmith tracing.
